In [0]:
from pyspark.sql import functions as F

In [0]:
#Configure the dataset
SEED = 20261022
EVENT_COUNT = 10000
CUSTOMER_COUNT = 1000
PRODUCT_COUNT = 20
DATA_END_DATE = "2026-11-29"  # analysis date (2026-10-30) + 30-day outcome window

print("Random Seed:", SEED)
print("Engagement Events:", EVENT_COUNT)

Random Seed: 20261022
Engagement Events: 10000


In [0]:
#Generate engagement events
from pyspark.sql import functions as F
from pyspark.sql.window import Window

events = spark.range(1, EVENT_COUNT + 1)

# Generate unique event IDs
events = events.withColumn(
    "event_id",
    F.concat(
        F.lit("EVN-2026-"),
        F.lpad(F.col("id").cast("string"), 8, "0")
    )
)

# Generate deterministic customer mapping
customer_lookup = (
    spark.table("customer_master")
    .select("customer_id", "signup_date")
    .withColumn(
        "customer_number",
        F.row_number().over(
            Window.orderBy("customer_id")
        )
    )
)

events = events.withColumn(
    "customer_number",
    (
        F.pmod(
            F.xxhash64("id", F.lit(SEED)),
            F.lit(CUSTOMER_COUNT)
        ) + 1
    ).cast("int")
)

events = events.join(
    customer_lookup,
    on="customer_number",
    how="inner"
)

# Generate event types
event_hash = F.pmod(
    F.xxhash64("id", F.lit("event")),
    F.lit(100)
)

events = events.withColumn(
    "event_type",
    F.when(event_hash < 30, "WEBSITE_VISIT")
     .when(event_hash < 55, "PRODUCT_VIEW")
     .when(event_hash < 70, "ADD_TO_CART")
     .when(event_hash < 82, "EMAIL_OPEN")
     .when(event_hash < 90, "EMAIL_CLICK")
     .when(event_hash < 96, "PURCHASE")
     .otherwise("SUPPORT_PAGE_VIEW")
)

# Generate channels
events = events.withColumn(
    "channel",
    F.when(F.col("event_type").isin("WEBSITE_VISIT", "PRODUCT_VIEW", "ADD_TO_CART"), "WEBSITE")
     .when(F.col("event_type").isin("EMAIL_OPEN", "EMAIL_CLICK"), "EMAIL")
     .when(F.col("event_type") == "PURCHASE", "MOBILE_APP")
     .otherwise("SUPPORT_PORTAL")
)

# Generate product IDs for product-related events
product_lookup = (
    spark.table("product_master")
    .select("product_id")
    .withColumn(
        "product_number",
        F.row_number().over(Window.orderBy("product_id"))
    )
)

events = events.withColumn(
    "product_number",
    (
        F.pmod(
            F.xxhash64("id", F.lit("product")),
            F.lit(PRODUCT_COUNT)
        ) + 1
    ).cast("int")
)

events = events.join(
    product_lookup,
    on="product_number",
    how="left"
)

events = events.withColumn(
    "product_id",
    F.when(
        F.col("event_type").isin(
            "PRODUCT_VIEW", "ADD_TO_CART", "PURCHASE"
        ),
        F.col("product_id")
    ).otherwise(F.lit(None).cast("string"))
)

# Generate timestamps between the customer's signup date and DATA_END_DATE
seconds_available = (
    F.datediff(F.lit(DATA_END_DATE).cast("date"), F.col("signup_date")) + 1
) * 86400

events = events.withColumn(
    "event_timestamp",
    (
        F.col("signup_date").cast("timestamp").cast("long")
        + F.pmod(F.xxhash64("id", F.lit("time")), seconds_available)
    ).cast("timestamp")
)

# Generate campaign IDs for email interactions
events = events.withColumn(
    "campaign_id",
    F.when(
        F.col("event_type").isin("EMAIL_OPEN", "EMAIL_CLICK"),
        F.concat(
            F.lit("CMP-2026-"),
            F.lpad(
                (
                    F.pmod(
                        F.xxhash64("id", F.lit("campaign")),
                        F.lit(10)
                    ) + 1
                ).cast("string"),
                3,
                "0"
            )
        )
    ).otherwise(F.lit(None).cast("string"))
)

# Select final columns
engagement_df = events.select(
    "event_id",
    "customer_id",
    "event_type",
    "event_timestamp",
    "channel",
    "product_id",
    "campaign_id"
)

display(engagement_df.limit(10))

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1155: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(
/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1155: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


event_id,customer_id,event_type,event_timestamp,channel,product_id,campaign_id
EVN-2026-00000001,CUS-000415,PURCHASE,2026-09-13T06:02:56.000Z,MOBILE_APP,PRD-000011,null
EVN-2026-00000002,CUS-000422,WEBSITE_VISIT,2026-09-04T00:50:03.000Z,WEBSITE,null,null
EVN-2026-00000003,CUS-000705,WEBSITE_VISIT,2026-06-18T16:38:37.000Z,WEBSITE,null,null
EVN-2026-00000004,CUS-000715,PRODUCT_VIEW,2026-05-21T20:00:56.000Z,WEBSITE,PRD-000015,null
EVN-2026-00000005,CUS-000792,ADD_TO_CART,2026-08-26T16:09:13.000Z,WEBSITE,PRD-000008,null
EVN-2026-00000006,CUS-000785,PRODUCT_VIEW,2026-05-26T17:01:43.000Z,WEBSITE,PRD-000010,null
EVN-2026-00000007,CUS-000693,SUPPORT_PAGE_VIEW,2026-03-18T14:43:14.000Z,SUPPORT_PORTAL,null,null
EVN-2026-00000008,CUS-000880,WEBSITE_VISIT,2026-04-25T19:34:25.000Z,WEBSITE,null,null
EVN-2026-00000009,CUS-000151,WEBSITE_VISIT,2026-07-02T16:07:03.000Z,WEBSITE,null,null
EVN-2026-00000010,CUS-000103,ADD_TO_CART,2026-07-21T00:17:02.000Z,WEBSITE,PRD-000017,null


#####Validate engagement data


In [0]:
#A. Check total records
print("Total Events:", engagement_df.count())

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1155: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


Total Events: 10000


In [0]:
#Check duplicate event IDs
engagement_df.groupBy("event_id") \
    .count() \
    .filter(F.col("count") > 1) \
    .show()

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1155: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


+--------+-----+
|event_id|count|
+--------+-----+
+--------+-----+



In [0]:
#Check invalid customer IDs
invalid_customers = engagement_df.join(
    spark.table("customer_master").select("customer_id"),
    on="customer_id",
    how="left_anti"
)

print("Invalid Customer IDs:", invalid_customers.count())

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1155: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


Invalid Customer IDs: 0


In [0]:
#Check invalid product IDs
invalid_products = (
    engagement_df
    .filter(F.col("product_id").isNotNull())
    .join(
        spark.table("product_master").select("product_id"),
        on="product_id",
        how="left_anti"
    )
)

print("Invalid Product IDs:", invalid_products.count())

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1155: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


Invalid Product IDs: 0


In [0]:
#Analyze event distribution
engagement_df.groupBy("event_type") \
    .count() \
    .orderBy(F.desc("count")) \
    .show()

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1155: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


+-----------------+-----+
|       event_type|count|
+-----------------+-----+
|    WEBSITE_VISIT| 2969|
|     PRODUCT_VIEW| 2479|
|      ADD_TO_CART| 1504|
|       EMAIL_OPEN| 1243|
|      EMAIL_CLICK|  851|
|         PURCHASE|  595|
|SUPPORT_PAGE_VIEW|  359|
+-----------------+-----+



In [0]:
#Save the EngagementEvent table
engagement_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("engagement_event")

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1155: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


In [0]:
%sql
SELECT * FROM engagement_event LIMIT 20;

event_id,customer_id,event_type,event_timestamp,channel,product_id,campaign_id
EVN-2026-00000001,CUS-000415,PURCHASE,2026-09-13T06:02:56.000Z,MOBILE_APP,PRD-000011,null
EVN-2026-00000002,CUS-000422,WEBSITE_VISIT,2026-09-04T00:50:03.000Z,WEBSITE,null,null
EVN-2026-00000003,CUS-000705,WEBSITE_VISIT,2026-06-18T16:38:37.000Z,WEBSITE,null,null
EVN-2026-00000004,CUS-000715,PRODUCT_VIEW,2026-05-21T20:00:56.000Z,WEBSITE,PRD-000015,null
EVN-2026-00000005,CUS-000792,ADD_TO_CART,2026-08-26T16:09:13.000Z,WEBSITE,PRD-000008,null
EVN-2026-00000006,CUS-000785,PRODUCT_VIEW,2026-05-26T17:01:43.000Z,WEBSITE,PRD-000010,null
EVN-2026-00000007,CUS-000693,SUPPORT_PAGE_VIEW,2026-03-18T14:43:14.000Z,SUPPORT_PORTAL,null,null
EVN-2026-00000008,CUS-000880,WEBSITE_VISIT,2026-04-25T19:34:25.000Z,WEBSITE,null,null
EVN-2026-00000009,CUS-000151,WEBSITE_VISIT,2026-07-02T16:07:03.000Z,WEBSITE,null,null
EVN-2026-00000010,CUS-000103,ADD_TO_CART,2026-07-21T00:17:02.000Z,WEBSITE,PRD-000017,null
